# 集群车辆路径问题 (cluVRP)

**类别:** 路径

来源: [https://www.hexaly.com/templates/clustered-vehicle-routing-cluvrp](https://www.hexaly.com/templates/clustered-vehicle-routing-cluvrp)


## 问题描述

**在集群车辆路径问题 (cluVRP)** 中,一组具有相同载货能力的运输车辆必须为已知需求的客户集群提供单一商品的配送服务。集群在事先已知,由彼此相邻的客户组成。所有客户必须被访问恰好一次。车辆从一个共同的配送中心出发并最终返回配送中心,每辆车服务的总需求量不能超过其载货能力。同一集群内的客户必须被一起服务。换句话说,当车辆访问某集群中的一个客户时,必须在离开该集群之前访问该集群中的所有其他客户。目标是最小化总行驶距离。

	

### 学到的要点

- 添加 [列表决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模每辆卡车访问集群的顺序以及每个集群内客户的访问顺序
- 添加 [partition](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html#n-ary-operators) 约束以确保所有集群都被访问
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算行驶距离


## 数据

我们提供的**集群车辆路径问题 (cluVRP)** 实例来自论文 [Exact Algorithms for the Clustered Vehicle Routing Problem](https://www.researchgate.net/publication/260941176_Exact_Algorithms_for_the_Clustered_Vehicle_Routing_Problem/)。它们遵循 [TSPLib 格式](http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/DOC.PS),具体如下:

- 节点数量跟在关键字 DIMENSION 之后(由于有一个仓库,客户数量等于节点数减 1)。
- 卡车载货能力跟在关键字 CAPACITY 之后。
- 卡车数量跟在关键字 VEHICLES 之后。
- 集群数量跟在关键字 GVRP_CAPACITY 之后。
- 关键字 NODE_COORD_SECTION 之后:对每个节点,给出其 ID 和 x、y 坐标。
- 关键字 GVRP_SET_SECTION 之后:对每个集群,给出其 ID 以及属于该集群的节点(以值 -1 结束)。
- 关键字 DEMAND_SECTION 之后:对每个集群,给出其 ID 和总需求(该集群内客户需求的总和)。


## 模型

集群车辆路径问题 (cluVRP) 的 Hexaly 模型使用列表决策变量。对于每辆卡车,我们定义一个列表变量来表示它访问的集群序列(`truckSequences`)。通过在所有这些列表上使用 [**partition**](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html#n-ary-operators) 约束,我们确保每个集群恰好由一辆卡车服务。然后我们使用第二组列表决策变量来建模卡车访问每个集群内客户的顺序(`clustersSequences`)。为了确保我们访问所有客户,我们将这些列表的大小约束为等于集群中的客户数。

每辆卡车交付的总数量通过 [**lambda 函数**](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 对所有访问过的集群应用 `sum` 算子来计算。注意,该求和中项的数量以及列表的大小在搜索过程中是变化的。我们将该数量约束为小于卡车的载货能力。

然后我们计算每个集群内部的行驶距离。使用另一个 [**lambda 函数**](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html),我们沿路径将从一个客户到下一个客户的距离累加起来。我们还记录每个集群中访问的第一个和最后一个节点(分别为 `initialNodes` 和 `endNodes`)。

然后我们可以通过累加以下各项来计算每辆卡车的行驶距离:

- 在每个访问过的集群内行驶的距离
- 从一个集群的最后一个客户到下一个集群的第一个客户的距离之和
- 从配送中心到第一个集群的第一个客户的距离
- 从最后一个集群的最后一个客户返回配送中心的距离。

最后,我们最小化总行驶距离。


## Python 实现


In [ ]:
from math import hypot
from pathlib import Path

from optagent import ModelBuilder, solve


def read_input_cvrp(filename):
    tokens = Path(filename).read_text().split()
    iterator = iter(tokens)
    values = {}
    while (token := next(iterator)) != "NODE_COORD_SECTION":
        if token in {"DIMENSION:", "VEHICLES:", "GVRP_SETS:", "CAPACITY:"}:
            values[token] = int(next(iterator))
    nb_customers = values["DIMENSION:"] - 1
    coordinates, depot = [None] * nb_customers, None
    for node in range(values["DIMENSION:"]):
        node_id, x, y = int(next(iterator)), float(next(iterator)), float(next(iterator))
        if node_id != node + 1:
            raise ValueError("node identifiers must be consecutive")
        if node == 0:
            depot = (x, y)
        else:
            coordinates[node - 1] = (x, y)
    if next(iterator) != "GVRP_SET_SECTION":
        raise ValueError("missing GVRP_SET_SECTION")
    clusters = []
    for cluster_id in range(values["GVRP_SETS:"]):
        if int(next(iterator)) != cluster_id + 1:
            raise ValueError("cluster identifiers must be consecutive")
        cluster = []
        while (customer := int(next(iterator))) != -1:
            cluster.append(customer - 2)
        clusters.append(cluster)
    if next(iterator) != "DEMAND_SECTION":
        raise ValueError("missing DEMAND_SECTION")
    demands = []
    for cluster_id in range(values["GVRP_SETS:"]):
        if int(next(iterator)) != cluster_id + 1:
            raise ValueError("demand identifiers must be consecutive")
        demands.append(int(next(iterator)))

    def distance(left, right):
        return int(hypot(left[0] - right[0], left[1] - right[1]) + 0.5)

    matrix = [[distance(left, right) for right in coordinates] for left in coordinates]
    depot_distances = [distance(depot, customer) for customer in coordinates]
    return values["VEHICLES:"], values["CAPACITY:"], matrix, depot_distances, demands, clusters


def main(instance_file, time_limit=20):
    nb_trucks, capacity, matrix_data, depot_data, demands_data, clusters_data = read_input_cvrp(instance_file)
    model = ModelBuilder()
    matrix, depot_distances = model.array(matrix_data), model.array(depot_data)
    demands = model.array(demands_data)

    cluster_sequences = []
    cluster_distances, initial_nodes, end_nodes = [], [], []
    for cluster_id, cluster in enumerate(clusters_data):
        sequence = model.list(len(cluster), name=f"cluster_{cluster_id}")
        model.constraint(sequence.count() == len(cluster))
        cluster_sequences.append(sequence)
        customers = model.array(cluster)
        cluster_distances.append(
            model.sum(
                *(
                    matrix[customers[sequence.at(position - 1)]][customers[sequence.at(position)]]
                    for position in range(1, len(cluster))
                )
            )
        )
        initial_nodes.append(customers[sequence.at(0)])
        end_nodes.append(customers[sequence.at(len(cluster) - 1)])

    cluster_distances_array = model.array(cluster_distances)
    initial_nodes_array, end_nodes_array = model.array(initial_nodes), model.array(end_nodes)
    truck_sequences = [
        model.list(len(clusters_data), name=f"truck_{truck}")
        for truck in range(nb_trucks)
    ]
    model.constraint(model.partition(truck_sequences))

    route_distances = []
    for sequence in truck_sequences:
        count = sequence.count()
        model.constraint(model.sum(sequence, model.lambda_function(lambda cluster: demands[cluster])) <= capacity)
        between_clusters = model.sum(
            model.range(1, count),
            model.lambda_function(
                lambda position: cluster_distances_array[sequence.at(position)]
                + matrix[end_nodes_array[sequence.at(position - 1)]][initial_nodes_array[sequence.at(position)]]
            ),
        )
        route_distances.append(
            model.iif(
                count > 0,
                between_clusters
                + cluster_distances_array[sequence.at(0)]
                + depot_distances[initial_nodes_array[sequence.at(0)]]
                + depot_distances[end_nodes_array[sequence.at(count - 1)]],
                0,
            )
        )
    total_distance = model.sum(*route_distances)
    model.minimize(total_distance, name="total_distance")
    solution = solve(model, time_limit_s=float(time_limit))
    values = solution.values(
        {"total_distance": total_distance, **{f"truck_{i}": route for i, route in enumerate(truck_sequences)}}
    )
    print(f"Total distance = {values['total_distance']}; Status = {solution.status.value}")
    for truck, route in enumerate(truck_sequences):
        customers = [
            clusters_data[cluster][customer] + 2
            for cluster in values[f"truck_{truck}"]
            for customer in solution.values({f"cluster_{cluster}": cluster_sequences[cluster]})[f"cluster_{cluster}"]
        ]
        print(f"Truck {truck + 1}: {' '.join(map(str, customers))}")
    return solution

In [ ]:
INSTANCE_DIR = Path.cwd() / "instances"
solution = main(INSTANCE_DIR / "A-n32-k5-C11-V2.gvrp", time_limit=5)